# 08 - Greeks

Delta, gamma, theta, vega, rho for the option chain.

Requires fetching with `--greeks`:
    python -m scripts_python.fetch_options --currency ETH --greeks


## Setup

In [ ]:
import sys, sqlite3
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / "database").exists():
    _p = _p.parent
sys.path.insert(0, str(_p))

OPT_DB = _p / "database" / "options.db"
MKT_DB = _p / "database" / "market.db"

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (13, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

def opt(q, **p):
    with sqlite3.connect(OPT_DB) as c:
        return pd.read_sql_query(q, c, params=p)

def mkt(q, **p):
    with sqlite3.connect(MKT_DB) as c:
        return pd.read_sql_query(q, c, params=p)

print("options.db:", OPT_DB)
print("market.db :", MKT_DB)

## 1. Load greeks

In [ ]:
g = opt("SELECT * FROM greeks WHERE underlying='ETH' ORDER BY ts DESC LIMIT 500")
print("rows:", len(g))
if g.empty:
    print("no greeks — fetch with --greeks flag")
g.head()

## 2. Delta by strike

In [ ]:
if not g.empty:
    snap = g[g.ts == g.ts.max()].copy()
    target = snap.expiry.iloc[0]
    s = snap[snap.expiry == target]

    fig, ax = plt.subplots()
    for kind, color in [("C", "seagreen"), ("P", "crimson")]:
        d = s[s["kind"] == kind].sort_values("strike")
        ax.plot(d.strike, d.delta, "o-", color=color,
                label="Call delta" if kind == "C" else "Put delta")
    ax.axhline(0, ls="--", color="black", lw=0.8)
    ax.set_title(f"delta by strike — {pd.to_datetime(target, unit='ms').date()}")
    ax.set_xlabel("strike")
    ax.set_ylabel("delta")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 3. Gamma

In [ ]:
if not g.empty:
    snap = g[g.ts == g.ts.max()].copy()
    target = snap.expiry.iloc[0]
    s = snap[snap.expiry == target].sort_values("strike")

    fig, ax = plt.subplots()
    ax.plot(s.strike, s.gamma, "o-", color="darkorange")
    ax.set_title("gamma by strike")
    ax.set_xlabel("strike")
    ax.set_ylabel("gamma")
    plt.tight_layout()
    plt.show()